In [ ]:
import duckdb
import pandas as pd

from pytrends.request import TrendReq


# 📌 2. DuckDB bağlantısı
con = duckdb.connect("/home/ubuntu/blog-factory/warehouse/blog_factory.duckdb", read_only=True) 

# Burada ideas tablosuna target_keywords alanını da eklediğini varsayıyorum
df_ideas = con.execute("""
    SELECT idea_id, idea_title, category_slug
           -- target_keywords kolonunu senin pipeline'a eklemen lazım
           -- şimdilik örnek amaçlı burada manuel ekleyeceğiz
    FROM ideas
    LIMIT 5;
""").df()

df_ideas



,idea_id,idea_title,category_slug
0,i-alternatives-to-traditional-nose-studs-explo...,Alternatives to Traditional Nose Studs: Explor...,beauty
1,i-faux-vs-real-which-eyelashes-give-you-the-be...,Faux vs. Real: Which Eyelashes Give You the Be...,beauty
2,i-the-refreshing-benefits-of-tea-tree-and-lemo...,The Refreshing Benefits of Tea Tree and Lemon ...,beauty
3,i-top-10-adorable-hair-accessories-for-kids-a-...,Top 10 Adorable Hair Accessories for Kids: A M...,beauty
4,i-the-story-behind-quality-how-premium-brushes...,The Story Behind Quality: How Premium Brushes ...,beauty


In [ ]:
# 📌 3. Test için target_keywords ekleyelim (normalde DB schema'na eklenecek)
# Basit şekilde ilk 5 fikir için elle dolduralım
manual_keywords = [
    "nose studs, alternative piercings",
    "eyelashes, faux eyelashes, real eyelashes",
    "tea tree oil, lemon oil benefits",
    "hair accessories for kids",
    "premium brushes, makeup brushes"
]

df_ideas["target_keywords"] = manual_keywords

# 📌 4. Pytrends init
pytrends = TrendReq(hl="en-US", tz=360)

# 📌 5. Kategori eşleme → Google Trends category ID
category_map = {
    "electronics": 5,
    "beauty": 44,
    "health": 45,
    "games": 8,
    "shopping": 18
}

# 📌 6. Trends score fonksiyonu
def trends_score(keyword, cat_id):
    try:
        pytrends.build_payload([keyword], cat=cat_id, timeframe="today 12-m", geo="US")
        data = pytrends.interest_over_time()
        if data.empty:
            return 0
        return int(data[keyword].mean())  # ortalama score
    except Exception:
        return 0


In [ ]:
def optimize_title(row):
    idea_title = row["idea_title"]
    target_keywords = row["target_keywords"].split(",")
    category_slug = row["category_slug"]

    main_kw = target_keywords[0].strip()
    cat_id = category_map.get(category_slug, 0)

    # 1. Ana keyword için trend check
    score = get_trend_score(main_kw, cat_id)

    # 2. Eğer skor çok düşükse, fallback → related_queries
    if score < 5:
        related = pytrends.related_queries()
        if main_kw in related and related[main_kw]["top"] is not None:
            candidates = related[main_kw]["top"]["query"].head(3).tolist()
            best_kw, best_score = None, 0
            for cand in candidates:
                s = get_trend_score(cand, cat_id)
                if s > best_score:
                    best_kw, best_score = cand, s
            if best_score >= 5:  # alternatif bulursak
                return rewrite_title(idea_title, best_kw), best_score, "Fallback"
    
    # 3. Normal case → skor ≥5
    return rewrite_title(idea_title, main_kw), score, priority_label(score)


In [ ]:
# 📌 8. Uygula
df_ideas[["optimized_title", "priority", "trend_score"]] = df_ideas.apply(optimize_title, axis=1)

df_ideas[["idea_title", "target_keywords", "optimized_title", "priority", "trend_score"]]


,idea_title,target_keywords,optimized_title,priority,trend_score
0,Alternatives to Traditional Nose Studs: Explor...,"nose studs, alternative piercings",Top nose studs Picks: Exploring New Trends,Medium,8
1,Faux vs. Real: Which Eyelashes Give You the Be...,"eyelashes, faux eyelashes, real eyelashes",Best eyelashes 2025: Which Eyelashes Give You ...,High,84
2,The Refreshing Benefits of Tea Tree and Lemon ...,"tea tree oil, lemon oil benefits",Best tea tree oil 2025: The Refreshing Benefit...,High,89
3,Top 10 Adorable Hair Accessories for Kids: A M...,hair accessories for kids,Top 10 Adorable Hair Accessories for Kids: A M...,Low,3
4,The Story Behind Quality: How Premium Brushes ...,"premium brushes, makeup brushes",The Story Behind Quality: How Premium Brushes ...,Low,0
